# CNN Training Notebook

### 0. Import Library & Dataset

In [ ]:
# Standard libraries
import os
import sys
import itertools
import time
import json

from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualisasi
import matplotlib.pyplot as plt

# Evaluasi Model
from sklearn.metrics import f1_score, classification_report

# Deep Learning
import tensorflow as tf
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

print('TF: ', tf.__version__)
print('GPU: ', tf.config.list_physical_devices('GPU'))

TF:  2.10.0
GPU:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
src = Path(os.getcwd()) # Path on this file
while not (src / 'src').exists() and src != src.parent:
    src = src.parent

sys.path.insert(0, str(src))
os.chdir(src)

In [ ]:
DATA_DIR = Path('data/flickr8k')
IMAGES_DIR = DATA_DIR / 'Images'
CAPS_CSV = DATA_DIR / 'captions.txt'
MODEL_DIR = Path('models/rnn_lstm')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEAT_CACHE  = MODEL_DIR / 'inception_features.npy'
VOCAB_PATH  = MODEL_DIR / 'vocab.json'

MAX_LEN    = 40
EMBED_DIM  = 256
BATCH_SIZE = 64
EPOCHS     = 20


### 1. Feature Extraction

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from src.cnn.utils import extract_features

cnn_encoder = InceptionV3(include_top=False, pooling='avg', weights='imagenet')
cnn_encoder.trainable=False

FEATURE_DIM = cnn_encoder.output_shape[-1]
print(f'Feature dim: {FEATURE_DIM}')

In [ ]:
all_image_paths = sorted(IMAGES_DIR.glob('*.jpg'))
print(f'Total images: {len(all_image_paths)}')

feat_file = MODEL_DIR / 'inception_features.npy'
feat_names_file = feat_file = MODEL_DIR / 'inception_features.npy'

if feat_file.exist() and feat_names_file.exists():
    feats_arr = np.load(feat_file)

    with open(feat_names_file) as f:
        feat_names = json.load(f)
        
    features = {}
    for name, feat in zip(feat_names, feats_arr):
        features[name] = feat

else:
    print('Extract feature process')

    paths_str = []
    for p in all_image_paths:
        paths_str.append(str(p))

    feats_arr = extract_features(paths_str, cnn_encoder, str(feat_file), target_size=(299, 299), preprocess_fn=preprocess_input, batch_size=64)
    
    feat_names = []
    for p in all_image_paths:
        feat_names.append(p.name)
    
    with open(feat_names_file, 'w') as f:
        json.dump(feat_names, f)

    features = {}
    for name, feat in zip(feat_names, feats_arr):
        features[name] = feat
    
    print(f'There is {len(features)} feats')

### 2. Caption Preprocessing

In [ ]:
from src.rnn_lstm.preprocess import (load_captions, 
                                     auto_split, 
                                     build_vocab, 
                                     save_vocab, 
                                     load_vocab, 
                                     make_sequences
                                     )

captions = load_captions(str(CAPS_CSV))
print(f'Total images: {len(captions)}')

train_imgs, val_imgs, test_imgs = auto_split(captions)
print(f'Split: {len(train_imgs)} train / {len(val_imgs)} val / {len(test_imgs)} test')


if VOCAB_PATH.exists():
    word2idx, idx2word = load_vocab(str(VOCAB_PATH))
else:
    word2idx = build_vocab(captions, train_imgs, min_freq=1)
    save_vocab(word2idx, str(VOCAB_PATH))

    idx2word = {}
    for k, v in word2idx.items():
        idx2word[v] = k

VOCAB_SIZE = len(word2idx)
print(f'Vocab size: {VOCAB_SIZE}')


In [ ]:
def make_tf_dataset(img_list, features_dict, captions_dict, word2idx, max_len, batch_size):
    imgs, cap_in, cap_tgt = make_sequences(captions_dict, img_list, word2idx, max_len)
    img_feats = np.stack([features_dict[img] for img in imgs], axis=0)

    weights = (cap_tgt != 0).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices(({'cap_input': cap_in, 'img_input': img_feats}, cap_tgt, weights)
                                            )
    return ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE), imgs, cap_tgt

train_ds, _, _  = make_tf_dataset(train_imgs, features, captions, word2idx, MAX_LEN, BATCH_SIZE)
val_ds, val_imgs_flat, val_cap_tgt = make_tf_dataset(val_imgs, features, captions, word2idx, MAX_LEN, BATCH_SIZE)
print('Dataset terbuat')

### 3. Decoder

In [ ]:
from tensorflow.keras.layers import(Input, Embedding, Dense, Reshape, Concatenate, LSTM, SimpleRNN, Lambda)
from tensorflow.keras.models import Model

def build_decoder(cell_type, num_layers, 
                  hidden_size, vocab_size=VOCAB_SIZE, 
                  embed_dim=EMBED_DIM, feature_dim=FEATURE_DIM, max_len=MAX_LEN):
    
    cap_in = Input(shape=(max_len,), name='cap_input')
    emb = Embedding(vocab_size, embed_dim, name='embedding')(cap_in)

    img_in = Input(shape=(feature_dim,), name='img_input')
    img_proj = Dense(embed_dim, name='dense_proj')(img_in)
    img_proj = Reshape((1, embed_dim))(img_proj)

    x = Concatenate(axis=1)([img_proj, emb])

    if cell_type == 'lstm':
        RNNLayer = LSTM
    else:
        RNNLayer = SimpleRNN

    for i in range(num_layers):
        x = RNNLayer(hidden_size, return_sequences=True,
                     name=f'{cell_type}_{i+1}')(x)

    x = Lambda(lambda t: t[:, 1:, :], name='slice_output')(x)
    out = Dense(vocab_size, activation='softmax', name='dense_out')(x)

    return Model([cap_in, img_in], out, name=f'dec_{cell_type}_L{num_layers}_H{hidden_size}')

In [ ]:
CONFIGS = list(itertools.product(['rnn', 'lstm'], [1, 2, 3], [128, 512]))
print(f'Total configs: {len(CONFIGS)}')
for ct, nl, hs in CONFIGS:
    print(f'  {ct:4s}  layers={nl}  hidden={hs}')

In [ ]:
train_histories = {}
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

for i, (cell_type, n_layers, h_size) in enumerate(CONFIGS):
    model_name = f'{cell_type}_L{n_layers}_H{h_size}'
    save_path = MODEL_DIR / f'{model_name}.h5'

    print(f'[{i+1:02d}/12] {model_name}', end=' ... ')

    if save_path.exists():
        continue
    else:
        model = build_decoder(cell_type, n_layers, h_size)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

        hist = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=[early_stop], verbose=0)
        model.save(save_path)

        train_histories[model_name] = hist.history
        print(f'trained {len(hist.history["loss"])} epochs')

with open(MODEL_DIR / 'train_histories.json', 'w') as f:
    json.dump(train_histories, f)
print('\nSemua model tersimpan.')


NameError: name 'tf' is not defined

### 4. Test Case

####   A. Variasi Arsitektur

In [ ]:
from src.rnn_lstm.model import CaptioningFromScratch

def compute_bleu4(refs_corpus, hyps_corpus):
    smooth_func = SmoothingFunction().method1

    return corpus_bleu(refs_corpus, hyps_corpus, weights=(0.25,)*4, smoothing_function=smooth_func)

def compute_meteor_avg(refs_list, hyps_list):
    scores = []
    for refs, hyp in zip(refs_list, hyps_list):
        s = meteor_score([r.split() for r in refs], hyp.split())
        scores.append(s)

    return float(np.mean(scores))

def eval_model_scratch(model_path, img_list, features_dict, captions_dict, word2idx, idx2word, max_len=MAX_LEN):
    decoder = tf.keras.models.load_model(model_path)
    scratch = CaptioningFromScratch.from_keras(cnn_encoder, decoder, img_size=(299,299), preprocess_fn=preprocess_input)

    img_unique = list(dict.fromkeys(img_list))
    hyps = []
    refs = []

    for img in img_unique:
        feat = features_dict.get(img)
        
        if feat is None:
            continue

        caption = scratch.generate_from_feature(feat, word2idx, idx2word, max_len)
        ref_raw = captions_dict.get(img, [])

        hyps.append(caption.split())
        for ref in ref_raw:
            ref_lowered = ref.lower()
            ref_split = ref_lowered.split()
            refs.append(ref_split)

        bleu4 = compute_bleu4(refs, hyps)
        meteor = compute_meteor_avg(
            [[' '.join(t) for t in rs] for rs in refs],
            [' '.join(h) for h in hyps]
        )

        return bleu4, meteor, scratch
    
    print('Helper functions siap.')

In [ ]:
exp1_results = []
best_models  = {}

for ct, nl, hs in CONFIGS:
    model_name = f'{ct}_L{nl}_H{hs}'
    save_path  = MODEL_DIR / f'{model_name}.h5'
    print(f'Evaluating {model_name} ...', end=' ')

    bleu4, meteor, _ = eval_model_scratch(
        save_path, test_imgs, features, captions, word2idx, idx2word  # test set
    )
    exp1_results.append({
        'model': model_name, 'cell': ct,
        'n_layers': nl, 'hidden': hs,
        'bleu4': round(bleu4, 4), 'meteor': round(meteor, 4)
    })
    print(f'BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}')

df1 = pd.DataFrame(exp1_results).sort_values('bleu4', ascending=False).reset_index(drop=True)
display(df1)

best_rnn_row  = df1[df1.cell=='rnn'].iloc[0]
best_lstm_row = df1[df1.cell=='lstm'].iloc[0]
print(f'Best RNN : {best_rnn_row["model"]} (BLEU-4={best_rnn_row["bleu4"]:.4f})')
print(f'Best LSTM: {best_lstm_row["model"]} (BLEU-4={best_lstm_row["bleu4"]:.4f})')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, title) in zip(axes, [
    ('cell', 'Tipe Cell (RNN vs LSTM)'),
    ('n_layers', 'Jumlah Layer'),
    ('hidden',   'Hidden Size'),
]):
    grp = df1.groupby(col)['bleu4'].mean().sort_values(ascending=False)
    ax.bar(grp.index.astype(str), grp.values, color='coral')
    ax.set_title(title); ax.set_ylabel('Mean BLEU-4')
    ax.set_ylim(max(0, grp.min()-0.01), grp.max()+0.01)
    for j, v in enumerate(grp.values):
        ax.text(j, v+0.001, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Pengaruh Hyperparameter terhadap BLEU-4')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'exp1_hparam.png', bbox_inches='tight')
plt.show()

In [ ]:
try:
    with open(MODEL_DIR / 'train_histories.json') as f:
        loaded_hist = json.load(f)
    train_histories.update(loaded_hist)
except FileNotFoundError:
    pass

if train_histories:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in train_histories.items():
        ct = name.split('_')[0]
        ls = '-' if ct == 'lstm' else '--'
        ax1.plot(h['loss'],     label=name, linestyle=ls, alpha=0.7)
        ax2.plot(h['val_loss'], label=name, linestyle=ls, alpha=0.7)
    ax1.set_title('Train Loss'); ax1.legend(fontsize=6)
    ax2.set_title('Val Loss');   ax2.legend(fontsize=6)
    plt.suptitle('Loss Curves — Semua 12 Variasi')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'exp1_loss_curves.png', bbox_inches='tight')
    plt.show()

####   B. Keras v. Scratch

In [2]:
eval_imgs_exp2 = []

for img in test_imgs:
    if img in features:
        eval_imgs_exp2.append(img)

eval_imgs_exp2 = eval_imgs_exp2[:100]

# --
eval_feats_exp2 = []
temp_feat_list = []

for img in eval_imgs_exp2:
    feat = features[img]
    temp_feat_list.append(feat)

eval_feats_exp2 = np.stack(temp_feat_list)

# --
eval_refs_exp2 = []

for img in eval_imgs_exp2:
    caption = captions.get(img, [])

    result_caption = []

    for capt in caption:
        lowered = capt.lower()
        split = lowered.split()

        sentence = ' '.join(split)
        result_caption.append(sentence)

    eval_refs_exp2.append(result_caption)


rnn_path  = MODEL_DIR / f'{best_rnn_row["model"]}.h5'
lstm_path = MODEL_DIR / f'{best_lstm_row["model"]}.h5'
rnn_decoder_e2  = tf.keras.models.load_model(rnn_path)
lstm_decoder_e2 = tf.keras.models.load_model(lstm_path)

NameError: name 'test_imgs' is not defined

In [ ]:
rnn_scratch_e2 = CaptioningFromScratch.from_keras(cnn_encoder, rnn_decoder_e2, 
                                                  img_size=(299, 299), preprocess_fn=preprocess_input)

lstm_scratch_e2 = CaptioningFromScratch.from_keras(cnn_encoder, lstm_decoder_e2,
                                                   img_size=(299, 299), preprocess_fn=preprocess_input)


time_start = time.time()
rnn_scratch_caps = rnn_scratch_e2.generate_batch(eval_feats_exp2, word2idx, idx2word, MAX_LEN)
time_rnn_scratch = time.time() - time_start

time_start = time.time()
lstm_scratch_caps = lstm_scratch_e2.generate_batch(eval_feats_exp2, word2idx, idx2word, MAX_LEN)
time_lstm_scratch = time.time() - time_start

print(f'RNN Scratch : {time_rnn_scratch:.2f}s')
print(f'LSTM Scratch : {time_lstm_scratch:.2f}s')

In [ ]:
def keras_greedy_decode(decoder_model, feat, word2idx, idx2word, max_len):
    start_id = word2idx['<start>']
    end_id = word2idx['<end>']

    tokens = np.zeros((1, max_len), dtype=np.int32)
    tokens[0, 0] = start_id
    feat_b = feat[np.newaxis]
    result = []

    for timestep in range(max_len -1):
        out = decoder_model.predict([tokens, feat_b], verbose = 0)
        next_token =  int(np.argmax(out[0, timestep]))

        if next_token == end_id:
            break
        
        result.append(next_token)
        tokens[0, timestep+1] = next_token
    
    list_words = []
    for r in result:
        word = idx2word.get(i, '<unk>')
        list_words.append(word)

    hasil = ' '.join(list_words)
    return hasil

time_start = time.time()
